# Modul 15: TensorFlow-Grundlagen und dichte Keras-Modelle | Lösungen

## Überblick

Sie untersuchen TensorFlow-Tensoren, Broadcasting, automatische Gradienten und tf.data-Pipelines. Danach erstellen, trainieren, regularisieren, bewerten und speichern Sie ein kleines dichtes Keras-Modell für tabellarische Daten.

**Zugehörige Vorlesungen**

- **TensorFlow Grundlagen**
- **Dichte Keras-Modelle**

## Lernziele

Nach der Bearbeitung können Sie:

- TensorFlow-Tensoren, Formen, Datentypen, NumPy-Konvertierung und Broadcasting sicher verwenden.
- Gradienten mit GradientTape berechnen und kleine tf.data-Datasets reproduzierbar verarbeiten.
- Sequential-Modelle mit passenden Ein- und Ausgaben kompilieren, trainieren, bewerten und speichern.

## Geprüfte Fähigkeiten

- Tensoren, Datentypen, Broadcasting und GradientTape
- tf.data mit shuffle, batch, map und prefetch
- Keras Sequential, compile, fit, History, Regularisierung, Baseline und Modellpersistenz

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle prüft TensorFlow und lädt den kleinen Brustkrebs-Datensatz aus scikit-learn. Die Daten werden reproduzierbar in Training, Validierung und Test geteilt und ohne Test-Leakage skaliert. Das Modell bleibt klein und läuft auf der kostenlosen Colab-CPU.

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

print("TensorFlow-Version:", tf.__version__)
print("Verfügbare GPUs:", tf.config.list_physical_devices("GPU"))

daten = load_breast_cancer()
X_gesamt = daten.data.astype("float32")
y_gesamt = daten.target.astype("float32")

X_train_roh, X_test_roh, y_train, y_test = train_test_split(
    X_gesamt,
    y_gesamt,
    test_size=0.20,
    stratify=y_gesamt,
    random_state=RANDOM_SEED,
)
X_train_roh, X_val_roh, y_train, y_val = train_test_split(
    X_train_roh,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=RANDOM_SEED,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_roh).astype("float32")
X_val = scaler.transform(X_val_roh).astype("float32")
X_test = scaler.transform(X_test_roh).astype("float32")

print("Train, Validierung, Test:", X_train.shape, X_val.shape, X_test.shape)

### Aufgabe 1: Tensoren, Formen, Datentypen und NumPy-Konvertierung

Erzeugen Sie aus der vorgegebenen Python-Liste einen Tensor mit `dtype=tf.float32`. Geben Sie Form, Rang und Datentyp aus. Wandeln Sie ihn in ein NumPy-Array zurück und prüfen Sie Werte und Form.

Erzeugen Sie außerdem einen Integer-Tensor und zeigen Sie, dass für eine Division zunächst eine explizite Typumwandlung sinnvoll ist.

In [ ]:
python_matrix = [[1.0, 2.5, -1.0], [0.0, 4.0, 3.5]]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

tensor_float = tf.constant(python_matrix, dtype=tf.float32)
print("Tensor:\n", tensor_float)
print("Form:", tensor_float.shape)
print("Rang:", tf.rank(tensor_float).numpy())
print("Datentyp:", tensor_float.dtype)

numpy_matrix = tensor_float.numpy()
print("NumPy-Typ:", type(numpy_matrix).__name__)
print("NumPy-Form:", numpy_matrix.shape)
print("Werte identisch:", np.allclose(numpy_matrix, np.asarray(python_matrix)))

integer_tensor = tf.constant([[1, 2], [3, 4]], dtype=tf.int32)
# Durch tf.cast() wird deutlich festgelegt, dass eine Fließkommadivision gewünscht ist.
haelfte = tf.cast(integer_tensor, tf.float32) / 2.0
print("Integer-Tensor:", integer_tensor.dtype)
print("Ergebnis nach Cast:", haelfte.dtype)
print(haelfte.numpy())

> **Musterantwort und Interpretation**
>
> Schichten erwarten bestimmte Achsen und Datentypen. Eine vertauschte Achse kann inhaltlich falsche Berechnungen erzeugen, während ein unpassender Integer- oder Floattyp zu Fehlern, unnötigem Speicherverbrauch oder unerwarteter Genauigkeit führt. Frühe Prüfungen machen solche Probleme sichtbar, bevor sie in einer längeren Trainingsschleife schwer zu diagnostizieren sind.

### Aufgabe 2: Broadcasting mit TensorFlow nachvollziehen

Erzeugen Sie einen Tensor `bilder_batch` der Form `(2, 3, 4, 3)`, der zwei kleine RGB-Bilder repräsentiert. Ziehen Sie den Kanalvektor `[10, 20, 30]` per Broadcasting von jedem Pixel ab.

Prüfen Sie die resultierende Form und vergleichen Sie einen ausgewählten Pixel vor und nach der Operation. Erzeugen Sie außerdem ein absichtlich inkompatibles Gegenbeispiel und fangen Sie den erwarteten Fehler mit `try/except` ab.

In [ ]:
bilder_batch = tf.reshape(tf.range(2 * 3 * 4 * 3, dtype=tf.float32), (2, 3, 4, 3))
kanal_mittel = tf.constant([10.0, 20.0, 30.0], dtype=tf.float32)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

zentrierte_bilder = bilder_batch - kanal_mittel
print("Eingabeform:", bilder_batch.shape)
print("Ausgabeform:", zentrierte_bilder.shape)
print("Pixel vorher:", bilder_batch[1, 2, 3].numpy())
print("Pixel nachher:", zentrierte_bilder[1, 2, 3].numpy())

# Die letzte Achse der Bilder hat Länge 3. Daher passt ein Vektor der Form (3,).
assert zentrierte_bilder.shape == bilder_batch.shape
assert np.allclose(
    zentrierte_bilder[1, 2, 3].numpy(),
    bilder_batch[1, 2, 3].numpy() - kanal_mittel.numpy(),
)

try:
    inkompatibler_vektor = tf.constant([1.0, 2.0], dtype=tf.float32)
    _ = bilder_batch - inkompatibler_vektor
except (tf.errors.InvalidArgumentError, ValueError) as fehler:
    print("Erwarteter Formfehler erkannt:", type(fehler).__name__)

> **Musterantwort und Interpretation**
>
> Die Dimensionen werden von rechts nach links verglichen. Zwei Achsen sind kompatibel, wenn sie gleich groß sind oder eine von ihnen Länge 1 besitzt. Fehlende linke Achsen werden wie Achsen der Länge 1 behandelt. Der Kanalvektor passt daher zur letzten RGB-Achse und wird über Batch, Höhe und Breite wiederverwendet.

### Aufgabe 3: GradientTape mit manuellem Gradienten vergleichen

Für die Funktion

`L(w) = mean((w * x - y)^2)`

soll der Gradient nach dem skalaren Gewicht `w` berechnet werden. Verwenden Sie `tf.GradientTape` und leiten Sie den Gradienten zusätzlich mit NumPy manuell her. Führen Sie danach fünf Gradientenschritte mit Lernrate 0.1 aus und speichern Sie Gewicht und Verlust nach jedem Schritt.

In [ ]:
x_grad = tf.constant([1.0, 2.0, 3.0, 4.0], dtype=tf.float32)
y_grad = tf.constant([2.0, 4.0, 6.0, 8.0], dtype=tf.float32)
w = tf.Variable(0.5, dtype=tf.float32)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

with tf.GradientTape() as tape:
    vorhersage = w * x_grad
    verlust = tf.reduce_mean(tf.square(vorhersage - y_grad))
auto_gradient = tape.gradient(verlust, w)

# Für mean((w*x-y)^2) lautet dL/dw = mean(2*(w*x-y)*x).
x_np = x_grad.numpy()
y_np = y_grad.numpy()
w_np = float(w.numpy())
manueller_gradient = np.mean(2.0 * (w_np * x_np - y_np) * x_np)

print("Automatischer Gradient:", float(auto_gradient.numpy()))
print("Manueller Gradient:", float(manueller_gradient))
print("Abweichung:", abs(float(auto_gradient.numpy()) - manueller_gradient))

historie_gradient = []
lernrate = 0.1
for schritt in range(5):
    with tf.GradientTape() as tape:
        prognose = w * x_grad
        aktueller_verlust = tf.reduce_mean(tf.square(prognose - y_grad))
    gradient = tape.gradient(aktueller_verlust, w)
    w.assign_sub(lernrate * gradient)
    historie_gradient.append(
        {
            "Schritt": schritt + 1,
            "Gewicht": float(w.numpy()),
            "Verlust_vor_Update": float(aktueller_verlust.numpy()),
        }
    )

display(pd.DataFrame(historie_gradient).round(5))

> **Musterantwort und Interpretation**
>
> Eine Variable repräsentiert veränderbaren Zustand und kann durch Optimierer oder assign-Methoden aktualisiert werden. Konstanten bleiben unverändert. GradientTape kann zwar explizit beobachtete Tensoren ableiten, aber trainierbare Modellparameter werden üblicherweise als Variablen verwaltet und automatisch verfolgt.

### Aufgabe 4: tf.data mit Shuffling, Mapping und Batching aufbauen

Erstellen Sie aus `X_train` und `y_train` ein `tf.data.Dataset`. Mischen Sie mit festem Seed, verwenden Sie Batchgröße 32 und wenden Sie per `map` eine Funktion an, die jedem Merkmalsvektor eine zusätzliche letzte Achse gibt und das Label in `float32` umwandelt.

Nutzen Sie `prefetch(tf.data.AUTOTUNE)`. Inspizieren Sie Formen und Datentypen des ersten Batches. Erstellen Sie außerdem geordnete Validierungs- und Test-Datasets ohne Shuffling, jedoch ohne zusätzliche Achse, damit sie später zum dichten Modell passen.

In [ ]:
def erweitere_beispiel(merkmale, label):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def erweitere_beispiel(merkmale, label):
    # expand_dims(..., axis=-1) macht aus (30,) die Form (30, 1).
    return tf.expand_dims(merkmale, axis=-1), tf.cast(label, tf.float32)

train_dataset_demo = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset_demo = train_dataset_demo.shuffle(
    buffer_size=len(X_train),
    seed=RANDOM_SEED,
    reshuffle_each_iteration=True,
)
train_dataset_demo = train_dataset_demo.map(
    erweitere_beispiel,
    num_parallel_calls=tf.data.AUTOTUNE,
).batch(32).prefetch(tf.data.AUTOTUNE)

batch_merkmale, batch_labels = next(iter(train_dataset_demo))
print("Demo-Batch Merkmale:", batch_merkmale.shape, batch_merkmale.dtype)
print("Demo-Batch Labels:", batch_labels.shape, batch_labels.dtype)

# Das dichte Modell erwartet weiterhin Vektoren der Form (30,), nicht (30, 1).
train_dataset = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train), seed=RANDOM_SEED, reshuffle_each_iteration=True)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(64).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64).prefetch(tf.data.AUTOTUNE)

print("Element-Spezifikation Training:", train_dataset.element_spec)

> **Musterantwort und Interpretation**
>
> Die Reihenfolge beeinflusst eine aggregierte Kennzahl zwar meist nicht, aber geordnete Ausgaben erleichtern die Zuordnung von Vorhersagen zu Originalbeispielen und die Fehlersuche. Shuffling ist vor allem im Training nützlich, damit aufeinanderfolgende Mini-Batches nicht durch die ursprüngliche Datenordnung verzerrt werden.

### Aufgabe 5: Ein passendes Sequential-Modell erstellen und trainieren

Erstellen Sie ein kleines binäres Sequential-Modell mit expliziter Eingabeform, zwei Dense-Schichten und einer Sigmoid-Ausgabe. Verwenden Sie höchstens 16 und 8 verborgene Einheiten.

Kompilieren Sie mit Adam, binärer Kreuzentropie, Accuracy sowie AUC. Trainieren Sie höchstens 40 Epochen auf `train_dataset`, verwenden Sie `val_dataset` und EarlyStopping mit Wiederherstellung der besten Gewichte. Zeichnen Sie Trainings- und Validierungsverlust und vergleichen Sie Test-Accuracy sowie F1 mit einer Mehrheitsbaseline.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# MUSTERLÖSUNG
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

keras_modell = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],), name="merkmale"),
        layers.Dense(16, activation="relu", name="verborgen_1"),
        layers.Dense(8, activation="relu", name="verborgen_2"),
        layers.Dense(1, activation="sigmoid", name="wahrscheinlichkeit"),
    ],
    name="kleines_binaeres_mlp",
)
keras_modell.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)
keras_modell.summary()

fruehstopp = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=6,
    restore_best_weights=True,
)
history = keras_modell.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=40,
    callbacks=[fruehstopp],
    verbose=0,
)

history_tabelle = pd.DataFrame(history.history)
print("Ausgeführte Epochen:", len(history_tabelle))
display(history_tabelle.tail().round(4))

plt.plot(history_tabelle["loss"], label="Training")
plt.plot(history_tabelle["val_loss"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("Binäre Kreuzentropie")
plt.title("Keras-Trainingsverlauf")
plt.legend()
plt.show()

keras_wahrscheinlichkeit = keras_modell.predict(X_test, verbose=0).ravel()
keras_klasse = (keras_wahrscheinlichkeit >= 0.5).astype(int)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_klasse = baseline.predict(X_test)

keras_vergleich = pd.DataFrame(
    {
        "Modell": ["Mehrheitsbaseline", "Keras-MLP"],
        "Accuracy": [
            accuracy_score(y_test, baseline_klasse),
            accuracy_score(y_test, keras_klasse),
        ],
        "F1": [
            f1_score(y_test, baseline_klasse, zero_division=0),
            f1_score(y_test, keras_klasse, zero_division=0),
        ],
    }
)
display(keras_vergleich.round(3))

> **Musterantwort und Interpretation**
>
> Sigmoid liefert für jedes Beispiel eine einzelne Wahrscheinlichkeit zwischen null und eins. Binary Crossentropy bewertet genau diese Wahrscheinlichkeit gegen ein binäres Ziel. Für eine Mehrklassenaufgabe mit genau einer wahren Klasse wären dagegen typischerweise mehrere Softmax-Ausgaben und eine kategoriale Kreuzentropie passend.

### Aufgabe 6: Integrationsaufgabe: Regularisierung, Vorhersagen und Speichern

Erstellen Sie ein zweites Modell mit derselben Grundstruktur, ergänzen Sie aber L2-Regularisierung in den Dense-Schichten und Dropout nach der ersten verborgenen Schicht. Trainieren Sie mit denselben Daten und EarlyStopping.

Vergleichen Sie Parameterzahl, besten Validierungsverlust und Test-F1 beider Modelle. Speichern Sie das bessere Modell in einem temporären Verzeichnis im `.keras`-Format, laden Sie es neu und prüfen Sie, ob die ersten zehn Wahrscheinlichkeiten vor und nach dem Laden übereinstimmen.

In [ ]:
from tensorflow.keras import regularizers

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Das bereits trainierte Vergleichsmodell bleibt erhalten. Nur der Zufallsseed
# für die neue Modellinitialisierung wird erneut festgelegt.
tf.keras.utils.set_random_seed(RANDOM_SEED)

regularisiertes_modell = keras.Sequential(
    [
        keras.Input(shape=(X_train.shape[1],)),
        layers.Dense(
            16,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-3),
        ),
        layers.Dropout(0.20, seed=RANDOM_SEED),
        layers.Dense(
            8,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-3),
        ),
        layers.Dense(1, activation="sigmoid"),
    ],
    name="regularisiertes_mlp",
)
regularisiertes_modell.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

history_reg = regularisiertes_modell.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=40,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

reg_wahrscheinlichkeit = regularisiertes_modell.predict(X_test, verbose=0).ravel()
reg_klasse = (reg_wahrscheinlichkeit >= 0.5).astype(int)

modellvergleich = pd.DataFrame(
    {
        "Modell": ["Ohne explizite Regularisierung", "L2 + Dropout"],
        "Parameter": [keras_modell.count_params(), regularisiertes_modell.count_params()],
        "Bester_Val_Verlust": [min(history.history["val_loss"]), min(history_reg.history["val_loss"])],
        "Test_F1": [
            f1_score(y_test, keras_klasse),
            f1_score(y_test, reg_klasse),
        ],
    }
)
display(modellvergleich.round(4))

if modellvergleich.loc[1, "Test_F1"] >= modellvergleich.loc[0, "Test_F1"]:
    zu_speichern = regularisiertes_modell
else:
    zu_speichern = keras_modell

referenz = zu_speichern.predict(X_test[:10], verbose=0)
with tempfile.TemporaryDirectory() as temp_ordner:
    modell_pfad = os.path.join(temp_ordner, "tabellenmodell.keras")
    zu_speichern.save(modell_pfad)
    neu_geladen = keras.models.load_model(modell_pfad)
    nach_laden = neu_geladen.predict(X_test[:10], verbose=0)
    print("Vorhersagen stimmen überein:", np.allclose(referenz, nach_laden, atol=1e-6))
    assert np.allclose(referenz, nach_laden, atol=1e-6)

> **Musterantwort und Interpretation**
>
> Die vollständige Vorverarbeitung, hier insbesondere der angepasste StandardScaler, gehört zum reproduzierbaren Vorhersageweg. Außerdem sollten Merkmalsreihenfolge, Klassenbedeutung, Schwellenwert, Bibliotheksversionen, Trainingsdatenbeschreibung, Metriken und bekannte Grenzen dokumentiert werden. Ein gespeichertes Netz allein verhindert keine falsch geordneten oder anders skalierten Eingaben.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?